In [6]:
import pandas as pd
import numpy as np
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
import os


data = {
    "TransactionID": [
        "T1","T1","T1",
        "T2","T2","T2",
        "T3","T3",
        "T4","T4","T4","T4",
        "T5","T5","T5",
        "T6","T6",
        "T7","T7","T7",
        "T8","T8",
        "T9","T9","T9",
        "T10","T10","T10"
    ],
    "Item": [
        "Bread","Butter","Milk",
        "Bread","Butter","Eggs",
        "Milk","Cereal",
        "Bread","Butter","Jam","Eggs",
        "Milk","Cereal","Banana",
        "Bread","Butter",
        "Pasta","Pasta Sauce","Cheese",
        "Coffee","Sugar",
        "Diapers","Beer","Chips",
        "Bread","Milk","Eggs"
    ]
}

df = pd.DataFrame(data)

print("Raw transactions sample:")
print(df.head(15))
print("\nNumber of transactions (rows):", len(df))
print("Number of unique items:", df["Item"].nunique())



# Basic cleaning
df = df.dropna(subset=["TransactionID", "Item"])
df["Item"] = df["Item"].astype(str).str.strip().str.title()



# Convert to list of items per transaction
transactions = df.groupby("TransactionID")["Item"].apply(list).tolist()

print("\nTransactions as list of lists (first 5):")
print(transactions[:5])

# One-hot encode for mlxtend
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
transaction_df = pd.DataFrame(te_ary, columns=te.columns_)

print("\nOne-hot encoded transaction DataFrame shape:", transaction_df.shape)
print("Items (columns):", list(transaction_df.columns))


# Find frequent itemsets
min_support = 0.2  # at least 20% of transactions
frequent_itemsets = apriori(transaction_df, min_support=min_support, use_colnames=True)

print("\nFrequent itemsets (min_support = {}):".format(min_support))
print(frequent_itemsets)

# Generate association rules
min_confidence = 0.4
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=min_confidence
)

print("\nAssociation rules (min_confidence = {}):".format(min_confidence))
print(rules[["antecedents", "consequents", "support", "confidence", "lift"]])

# Optional: filter by lift > 1
rules_filtered = rules[rules["lift"] > 1].sort_values("lift", ascending=False)
print("\nRules with lift > 1 (sorted by lift):")
print(rules_filtered[["antecedents", "consequents", "support", "confidence", "lift"]])



# Sort by support
frequent_itemsets_sorted = frequent_itemsets.sort_values("support", ascending=False)

print("\nTop frequent itemsets by support:")
print(frequent_itemsets_sorted[["itemsets", "support"]].head(15))

# Create 'outputs' directory if it doesn't exist
output_dir = "outputs"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Save to CSV for report
frequent_itemsets_sorted.to_csv(os.path.join(output_dir, "frequent_itemsets.csv"), index=False)
rules.to_csv(os.path.join(output_dir, "rules.csv"), index=False)


def recommend_items(basket_items, rules_df, top_k=5):
    """
    Given a basket (list of items), recommend additional items
    based on association rules.
    """
    basket = set(basket_items)
    candidates = []

    for _, row in rules_df.iterrows():
        antecedent = set(row["antecedents"])
        consequent = set(row["consequents"])
        # Only consider rules where antecedent is fully in basket
        # and consequent has at least one new item
        if antecedent.issubset(basket) and not consequent.issubset(basket):
            new_items = consequent - basket
            for item in new_items:
                candidates.append((item, row["confidence"], row["lift"]))

    # Remove duplicates (keep best confidence, then lift)
    best = {}
    for item, conf, lift in candidates:
        if item not in best or (conf, lift) > best[item]:
            best[item] = (conf, lift)

    recs = [(item, conf, lift) for item, (conf, lift) in best.items()]
    recs.sort(key=lambda x: (x[2], x[1]), reverse=True)  # by lift, then confidence
    return recs[:top_k]

# Example usage:
sample_baskets = [
    ["Bread", "Butter"],
    ["Milk", "Cereal"],
    ["Pasta"],
    ["Diapers"],
    ["Coffee"]
]

print("\nSample recommendations:")
for basket in sample_baskets:
    recs = recommend_items(basket, rules_filtered)
    print(f"Basket: {basket}")
    if recs:
        for item, conf, lift in recs:
            print(f"  → Recommend: {item} (confidence={conf:.2f}, lift={lift:.2f})")
    else:
        print("  → No strong recommendations from rules.")
    print()




print("Key insights you can highlight in your report:")
print("- Items frequently bought together (top frequent itemsets).")
print("- Strong association rules (high confidence & lift).")
print("- Possible product placement changes (e.g., place items from same rule close together).")
print("- Bundle/promotion ideas (e.g., 'Buy X, get Y discounted').")
print("- Cross-sell logic for app/website ('Customers also bought ...').")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Raw transactions sample:
   TransactionID    Item
0             T1   Bread
1             T1  Butter
2             T1    Milk
3             T2   Bread
4             T2  Butter
5             T2    Eggs
6             T3    Milk
7             T3  Cereal
8             T4   Bread
9             T4  Butter
10            T4     Jam
11            T4    Eggs
12            T5    Milk
13            T5  Cereal
14            T5  Banana

Number of transactions (rows): 28
Number of unique items: 15

Transactions as list of lists (first 5):
[['Bread', 'Butter', 'Milk'], ['Bread', 'Milk', 'Eggs'], ['Bread', 'Butter', 'Eggs'], ['Milk', 'Cereal'], ['Bread', 'Butter', 'Jam', 'Eggs']]

One-hot encoded transaction DataFrame shape: (10, 15)
Items (columns): ['Banana', 'Beer', 'Bread', 'Butter', 'Cereal', 'Cheese', 'Chips', 'Coffee', 'Diapers', 'Eggs', 'Jam', 'Milk', 'Pasta', 'Pasta Sauce', 'Sugar']

Frequent itemsets (min_support = 0.2):
    support               itemsets
0       0.5                (Bread)
1  

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag